# Train KiwiLM Model D on Google Colab

This notebook trains Model D (`cnn_attention_mamba`) at approximately the Chinchilla 20:1 token-to-parameter ratio. It uses KiwiLM's existing CLI and stores prepared data and checkpoints in Google Drive.

Before starting, select **Runtime → Change runtime type → GPU**. An A100 or L4 is preferred. A T4 can run the model, but the portable PyTorch selective scan is intentionally a readable reference implementation rather than a fused Mamba kernel, so training will be slower.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Prepared data and checkpoints are kept in Drive so they survive a Colab runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Clone KiwiLM and install it with uv

In [ ]:
%cd /content
!test -d KiwiLM/.git || git clone https://github.com/Tasty-Kiwi/KiwiLM.git
%cd /content/KiwiLM
!git pull --ff-only
!pip install -q uv
!uv sync --frozen

In [ ]:
!uv run python -c "import torch; print('torch:', torch.__version__); print('cuda:', torch.cuda.is_available()); print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"

## 4. Verify Model D

In [ ]:
from kiwilm.config import CNNAttentionMambaConfig
from kiwilm.models import build_model

model_d = build_model(CNNAttentionMambaConfig())
parameter_count = sum(parameter.numel() for parameter in model_d.parameters())
print(f"Model D parameters: {parameter_count:,}")
assert parameter_count == 6_027_648
del model_d

## 5. Prepare or restore the 550k-story dataset

Model D has 6,027,648 parameters, so the 20:1 target is 120,552,960 training tokens. The current tokenizer averages about 222.2 tokens per story; 550k stories provides a margin over the estimated requirement.

The first run streams TinyStories and trains a new 8k byte-level BPE tokenizer. The prepared artifacts are then copied to Drive. Later Colab sessions restore those exact files instead of rebuilding them.

In [ ]:
%%bash
set -euo pipefail
cd /content/KiwiLM
DRIVE_DATA=/content/drive/MyDrive/KiwiLM/data/tinystories-550k
LOCAL_DATA=/content/KiwiLM/data/tinystories-550k
mkdir -p /content/drive/MyDrive/KiwiLM/data /content/KiwiLM/data

if [[ -f "$DRIVE_DATA/metadata.json" ]]; then
  echo "Restoring prepared data from Google Drive..."
  rm -rf "$LOCAL_DATA"
  cp -a "$DRIVE_DATA" /content/KiwiLM/data/
elif [[ ! -f "$LOCAL_DATA/metadata.json" ]]; then
  uv run kiwilm prepare \
    --output-dir "$LOCAL_DATA" \
    --train-limit 550000 \
    --validation-limit 10000 \
    --vocab-size 8192
  echo "Saving prepared data to Google Drive..."
  cp -a "$LOCAL_DATA" /content/drive/MyDrive/KiwiLM/data/
fi

du -sh "$LOCAL_DATA"
test -f "$LOCAL_DATA/metadata.json"

## 6. Start a new training run

Microbatch 8 × gradient accumulation 4 gives an effective batch of 32. With context 256, 14,716 optimizer steps process 120,553,472 tokens, or 20.0001 tokens per parameter. The smaller microbatch bounds the Mamba block's activation memory.

Run this cell only for a new run. It refuses to overwrite an existing Drive checkpoint. Checkpoints are saved every 500 steps, so a disconnect loses at most 500 optimizer steps.

In [ ]:
%%bash
set -euo pipefail
cd /content/KiwiLM
RUN_DIR=/content/drive/MyDrive/KiwiLM/runs/model-d-chinchilla
mkdir -p "$RUN_DIR"

if [[ -f "$RUN_DIR/latest.pt" ]]; then
  echo "An existing checkpoint was found. Run the resume cell instead."
  exit 1
fi

TORCH_ALLOW_TF32_CUBLAS_OVERRIDE=1 uv run kiwilm train \
  --architecture cnn_attention_mamba \
  --data-dir /content/KiwiLM/data/tinystories-550k \
  --output-dir "$RUN_DIR" \
  --device cuda \
  --batch-size 8 \
  --grad-accum-steps 4 \
  --max-steps 14716 \
  --warmup-steps 736 \
  --eval-interval 500 \
  --eval-batches 50 \
  --checkpoint-interval 500 \
  --log-interval 10 \
  --seed 42

## 7. Resume after a disconnect

In a fresh Colab session, rerun the GPU, Drive, setup, verification, and dataset-restore cells first. Then run this cell instead of the new-training cell. The CLI restores the optimizer, scheduler, random-generator, and batch-sampler state.

In [ ]:
%%bash
set -euo pipefail
cd /content/KiwiLM
RUN_DIR=/content/drive/MyDrive/KiwiLM/runs/model-d-chinchilla
test -f "$RUN_DIR/latest.pt"

TORCH_ALLOW_TF32_CUBLAS_OVERRIDE=1 uv run kiwilm train \
  --architecture cnn_attention_mamba \
  --data-dir /content/KiwiLM/data/tinystories-550k \
  --output-dir "$RUN_DIR" \
  --resume "$RUN_DIR/latest.pt" \
  --device cuda \
  --batch-size 8 \
  --grad-accum-steps 4 \
  --max-steps 14716 \
  --warmup-steps 736 \
  --eval-interval 500 \
  --eval-batches 50 \
  --checkpoint-interval 500 \
  --log-interval 10 \
  --seed 42

## 8. Evaluate the best checkpoint

In [ ]:
!uv run kiwilm evaluate --data-dir /content/KiwiLM/data/tinystories-550k --checkpoint /content/drive/MyDrive/KiwiLM/runs/model-d-chinchilla/best.pt --device cuda --batch-size 8 --batches 200 --seed 42

## 9. Generate a streamed sample

In [ ]:
!uv run kiwilm generate --data-dir /content/KiwiLM/data/tinystories-550k --checkpoint /content/drive/MyDrive/KiwiLM/runs/model-d-chinchilla/best.pt --device cuda --prompt "Once upon a time" --max-new-tokens 160 --temperature 0.6 --top-k 40 --seed 42 --stream

## Artifacts

The final run is stored under `MyDrive/KiwiLM/runs/model-d-chinchilla/`:

- `best.pt` — lowest validation-loss checkpoint
- `latest.pt` — most recent resumable checkpoint
- `metrics.jsonl` — training and validation history

The prepared dataset is stored under `MyDrive/KiwiLM/data/tinystories-550k/` and can be reused by the Model C notebook for a controlled comparison.